In [73]:
import polars as pl

In [74]:
def show_all(df, width=200, max_col_width=True):
    '''
    Prints an entire polars dataframe in the console or notebook output.
    Parameters
    ----------
    df : pl.DataFrame
        The dataframe to be printed.
    width : int, optional
        The width of the printed dataframe.
        Defaults to 200.
    max_col_width : bool, optional
        Whether to set the maximum column width.
        i.e. it will print the full contents of the cells.
        Defaults to True.
    '''
    with  pl.Config()  as  cfg:
        cfg.set_tbl_cols(-1)
        cfg.set_tbl_rows(-1)
        cfg.set_tbl_width_chars(width)
        if  max_col_width  or  len(df.columns) ==  1:
            cfg.set_fmt_str_lengths(width)
        print(df)

# batch 8

In [75]:
import extern
extern.run("grep SRA submit_batch3 |awk '{print $2}' |cat - runlists/batch1_500.txt runlists/batch2_500.txt runlists/batch4_2000.txt runlists/batch5_test10.txt runlists/batch6_test50000.txt runlists/batch7_test10000.txt >/tmp/runs")

''

In [76]:
prev = pl.read_csv("/tmp/runs", has_header=False)
prev.columns = ["acc"]
prev.shape

(63406, 1)

In [77]:
total = pl.read_csv('runlists/acc_less_than_20k_mbytes.csv', has_header=False)
total.columns = ["acc"]
total.shape

(710928, 1)

In [78]:
batch8_possible = total.join(prev, on="acc", how="anti")
batch8_possible.shape

(647522, 1)

In [79]:
# batch8 = batch8_possible.sample(100000)
# print(batch8.shape, batch8[:5])
# batch8.write_csv('runlists/batch8_100000.txt', include_header=False)

In [80]:
# 
extern.run("grep SRA submit_batch3 |awk '{print $2}' |cat - runlists/batch1_500.txt runlists/batch2_500.txt runlists/batch4_2000.txt runlists/batch5_test10.txt runlists/batch6_test50000.txt runlists/batch7_test10000.txt runlists/batch8_100000.txt >/tmp/runs")

prev = pl.read_csv("/tmp/runs", has_header=False)
prev.columns = ["acc"]
print(prev.shape)

batch9_possible = total.join(prev, on="acc", how="anti")
batch9_possible.shape

(163406, 1)


(547522, 1)

In [81]:
# batch9_possible.write_csv('runlists/batch9_da_rest.txt', include_header=False)

# batch10 - cleaning up and everythin 10Gbp or less

In [82]:
extern.run('cat  s3_us-east2.txt s3_woodcrob-sandpiper-us-east-1.txt |grep RR |sed "s/  */\t/g" >/tmp/runs')

prev1 = pl.read_csv('/tmp/runs', has_header=False, separator='\t')
prev1.columns = ["acc", "time", "size", "path"]
prev2 = prev1.with_columns(pl.col("path").alias("acc").str.split('.').list.get(0)).select(["acc", "size"])
prev2.shape, prev2[:3]

((137852, 2),
 shape: (3, 2)
 ┌───────────┬───────┐
 │ acc       ┆ size  │
 │ ---       ┆ ---   │
 │ str       ┆ i64   │
 ╞═══════════╪═══════╡
 │ DRR001961 ┆ 26759 │
 │ DRR003630 ┆ 60975 │
 │ DRR003636 ┆ 45914 │
 └───────────┴───────┘)

In [83]:
prev2.filter(pl.col("size") == 0).shape

(4305, 2)

In [84]:
prev3 = prev2.filter(pl.col("size") > 0)
prev3.shape

(133547, 2)

In [85]:
df = pl.read_csv('~/git/sandpiper/sra_metadata/sra_metadata_20250220.some_columns.csv.gz', has_header=False)
df.columns = ['acc','releasedate','mbases','mbytes']
df[:4]

acc,releasedate,mbases,mbytes
str,str,i64,i64
"""SRR15442735""","""2021-08-13T00:00:00+00:00""",6638,2614
"""ERR1959224""","""2017-07-08T00:00:00+00:00""",8555,3195
"""ERR5003368""","""2020-12-23T00:00:00+00:00""",1013,344
"""ERR5261058""","""2021-03-15T00:00:00+00:00""",20619,6792


In [86]:
df.shape, sum(df['mbases'])/1e9
#=> 780k metagenomes, 4.9 Pbases. A lot.

((783205, 4), 4.866907013)

In [87]:
batch10_possible = df.filter(pl.col('mbases') < 8000).join(prev, on="acc", how="anti")
batch10_possible.shape, batch10_possible.sample(3)

((471178, 4),
 shape: (3, 4)
 ┌────────────┬───────────────────────────┬────────┬────────┐
 │ acc        ┆ releasedate               ┆ mbases ┆ mbytes │
 │ ---        ┆ ---                       ┆ ---    ┆ ---    │
 │ str        ┆ str                       ┆ i64    ┆ i64    │
 ╞════════════╪═══════════════════════════╪════════╪════════╡
 │ SRR7234393 ┆ 2018-08-21T00:00:00+00:00 ┆ 7611   ┆ 4666   │
 │ ERR1293683 ┆ 2016-02-25T00:00:00+00:00 ┆ 1088   ┆ 675    │
 │ ERR537637  ┆ 2016-12-18T00:00:00+00:00 ┆ 6365   ┆ 4301   │
 └────────────┴───────────────────────────┴────────┴────────┘)

In [88]:
batch10_possible.select('acc').write_csv('runlists/batch10_10gbp_or_smaller.txt', include_header=False)

# batch11 small fast test of multiple

In [89]:
per_acc = pl.read_csv('~/m/msingle/mess/174_R220_renew/processing_20240531/per_acc_summary.csv')
per_acc.shape, per_acc[:3]

((245436, 11),
 shape: (3, 11)
 ┌────────────┬───────────┬───────────┬───────────┬───┬─────────┬───────────┬───────────┬───────────┐
 │ sample     ┆ root_cove ┆ species_c ┆ bacterial ┆ … ┆ warning ┆ predictio ┆ organism  ┆ host_or_n │
 │ ---        ┆ rage      ┆ overage   ┆ _archaeal ┆   ┆ ---     ┆ n         ┆ ---       ┆ ot        │
 │ str        ┆ ---       ┆ ---       ┆ _bases    ┆   ┆ str     ┆ ---       ┆ str       ┆ ---       │
 │            ┆ f64       ┆ f64       ┆ ---       ┆   ┆         ┆ str       ┆           ┆ str       │
 │            ┆           ┆           ┆ i64       ┆   ┆         ┆           ┆           ┆           │
 ╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═════════╪═══════════╪═══════════╪═══════════╡
 │ SRR8634435 ┆ 345.4     ┆ 0.933237  ┆ 117232064 ┆ … ┆ null    ┆ host      ┆ feces met ┆ host      │
 │            ┆           ┆           ┆ 2         ┆   ┆         ┆           ┆ agenome   ┆           │
 │ SRR8640623 ┆ 730.5     ┆ 0.127748  ┆ 139594647 ┆

In [90]:
batch11 = batch10_possible.join(per_acc.filter(pl.col('root_coverage')>0), left_on='acc', right_on='sample', how='inner').sort('mbases').head(4)

In [91]:
# batch11.select('acc').write_csv('runlists/batch11_4.txt', include_header=False)

# batch 12 - 8 to 30Gbp

In [92]:
df = pl.read_csv('~/git/sandpiper/sra_metadata/sra_metadata_20250220.some_columns.csv.gz', has_header=False)
df.columns = ['acc','releasedate','mbases','mbytes']
df[:4]

acc,releasedate,mbases,mbytes
str,str,i64,i64
"""SRR15442735""","""2021-08-13T00:00:00+00:00""",6638,2614
"""ERR1959224""","""2017-07-08T00:00:00+00:00""",8555,3195
"""ERR5003368""","""2020-12-23T00:00:00+00:00""",1013,344
"""ERR5261058""","""2021-03-15T00:00:00+00:00""",20619,6792


In [93]:
# how much 0-8, 8-15, 15-30 and 30+ mbases?
(
    df.select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') < 8000).select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') < 15000).filter(pl.col('mbases') >= 8000).select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') < 30000).filter(pl.col('mbases') >= 15000).select('mbases').sum()[0,0] /1e9,
    df.filter(pl.col('mbases') >= 30000).select('mbases').sum()[0,0] /1e9,
)


(4.866907013, 1.763913074, 1.36664454, 0.898161387, 0.838188012)

In [94]:
# Try those < 8gbases again where they failed - maybe the increased RAM of a this batch will get them through?
extern.run('cat s3_ls/* |grep RR |sed "s/  */\t/g" >/tmp/runs')

prev4 = pl.read_csv('/tmp/runs', has_header=False, separator='\t')
prev4.columns = ["acc", "time", "size", "path"]
print(prev4.group_by(pl.col('size') > 0).len())
prev5 = prev4.filter(pl.col('size') > 0).with_columns(pl.col("path").alias("acc").str.split('.').list.get(0))
prev5.shape

shape: (2, 2)
┌───────┬────────┐
│ size  ┆ len    │
│ ---   ┆ ---    │
│ bool  ┆ u32    │
╞═══════╪════════╡
│ false ┆ 8769   │
│ true  ┆ 591446 │
└───────┴────────┘


(591446, 4)

In [95]:
# Create a batch of 1000 runs with 8-30 Gbps, that aren't in any previous 
batch12_possible = df.filter(pl.col('mbases') < 30000).filter(pl.col('mbases') >= 8000).join(prev5, on="acc", how="anti")
batch12_possible.shape[0], batch12_possible.sample(5), batch12_possible.select(pl.col('mbases')).sum()[0,0] /1e9

(146031,
 shape: (5, 4)
 ┌─────────────┬───────────────────────────┬────────┬────────┐
 │ acc         ┆ releasedate               ┆ mbases ┆ mbytes │
 │ ---         ┆ ---                       ┆ ---    ┆ ---    │
 │ str         ┆ str                       ┆ i64    ┆ i64    │
 ╞═════════════╪═══════════════════════════╪════════╪════════╡
 │ SRR15275266 ┆ 2023-12-30T00:00:00+00:00 ┆ 8800   ┆ 2686   │
 │ SRR10823762 ┆ 2020-05-01T00:00:00+00:00 ┆ 14282  ┆ 5840   │
 │ SRR30235446 ┆ 2024-08-15T00:00:00+00:00 ┆ 16426  ┆ 6845   │
 │ ERR10775748 ┆ 2024-01-04T00:00:00+00:00 ┆ 13092  ┆ 4252   │
 │ SRR21474783 ┆ 2022-09-09T00:00:00+00:00 ┆ 17383  ┆ 12271  │
 └─────────────┴───────────────────────────┴────────┴────────┘,
 1.957546801)

In [96]:
# batch12_possible.sample(fraction=1, shuffle=True).select('acc').write_csv('runlists/batch12_8_to_30gbp.txt', include_header=False)

# batch 13 - 30gbp and above

In [97]:
batch13_possible = df.filter(pl.col('mbases') >= 30000).join(prev5, on="acc", how="anti")
batch13_possible.shape[0], batch13_possible.sample(5), batch13_possible.select(pl.col('mbases')).sum()[0,0] /1e9

(16011,
 shape: (5, 4)
 ┌─────────────┬───────────────────────────┬────────┬────────┐
 │ acc         ┆ releasedate               ┆ mbases ┆ mbytes │
 │ ---         ┆ ---                       ┆ ---    ┆ ---    │
 │ str         ┆ str                       ┆ i64    ┆ i64    │
 ╞═════════════╪═══════════════════════════╪════════╪════════╡
 │ SRR17275369 ┆ 2021-12-20T00:00:00+00:00 ┆ 30985  ┆ 10154  │
 │ SRR26046838 ┆ 2023-09-13T00:00:00+00:00 ┆ 55797  ┆ 19955  │
 │ SRR18067560 ┆ 2024-01-19T00:00:00+00:00 ┆ 79716  ┆ 26096  │
 │ SRR6837581  ┆ 2018-03-15T00:00:00+00:00 ┆ 31471  ┆ 17337  │
 │ SRR28974642 ┆ 2024-05-08T00:00:00+00:00 ┆ 34408  ┆ 9232   │
 └─────────────┴───────────────────────────┴────────┴────────┘,
 0.798983057)

In [98]:
# batch13_possible.sample(fraction=1, shuffle=True).select('acc').write_csv('runlists/batch13_30gbp_plus.txt', include_header=False)

# batch 14 - 30gbp and above retrying failed ones

In [99]:
# ➜  date; rclone ls s3:woodcrob-sandpiper-us-east-1/unannotated6/ >unannotated6.finished.ls.txt
# Mon 24 Mar 2025 09:27:23 AEST
# ➜  wc -l unannotated6.finished.ls.txt
#14262 unannotated6.finished.ls.txt

# extern.run('cat  unannotated6.finished.ls.txt |grep RR |sed "s/  */\t/g" >/tmp/runs')
# batch13_finished_samples = pl.read_csv("/tmp/runs", has_header=False, separator='\t')
# batch13_finished_samples.columns = ["null","size","acc1"]
# batch13_finished_samples.drop_in_place("null")
# batch13_finished_samples = batch13_finished_samples.with_columns(pl.col("acc1").str.split('.').list.get(0).alias("acc"))
# # batch13_finished_samples.columns = ["acc"]
# batch13_finished_samples.shape, batch13_finished_samples[:4], batch13_finished_samples.filter(pl.col("size") == 0).shape

In [100]:
# batch14_possible = batch13_possible.join(batch13_finished_samples.filter(pl.col("size") != 0), on="acc", how="anti")
# batch14_possible.shape, batch14_possible[:4]

In [101]:
# # show_all(batch14_possible.sample(10))
# batch14_possible.select(pl.col('mbytes').sum()) / 1e6
# # => 68 TB. Bit much to do locally.

In [102]:
# batch14_possible.sample(fraction=1).select('acc').write_csv('runlists/batch14_30gbp_plus_retries.txt', include_header=False)

# Batch 15 - chunking into 3 Gbp chunks

In [103]:
# ben in 🌐 b2 in sandpiper/sra_metadata/shotgun_sra_20250220 on  main [!?] via singlem-dev 20250324 took 48s 
# ➜  jq -rc '[.acc,.releasedate,.mbases,.mbytes,.organism,.avgspotlen, (.attributes[] | select(.k == "bases") | .v)] |@csv' <(cat *) |pigz >../sra_metadata_20250220.some_columns3.csv.gz
df = pl.read_csv('~/git/sandpiper/sra_metadata/sra_metadata_20250220.some_columns3.csv.gz', has_header=False)
df.columns = ['acc','releasedate','mbases','mbytes','organism','avespotlen','bases']
df[:4]

acc,releasedate,mbases,mbytes,organism,avespotlen,bases
str,str,i64,i64,str,i64,i64
"""SRR15442735""","""2021-08-13T00:00:00+00:00""",6638,2614,"""viral metagenome""",300,6638665200
"""ERR1959224""","""2017-07-08T00:00:00+00:00""",8555,3195,"""feces metagenome""",300,8555841630
"""ERR5003368""","""2020-12-23T00:00:00+00:00""",1013,344,"""gut metagenome""",299,1013578651
"""ERR5261058""","""2021-03-15T00:00:00+00:00""",20619,6792,"""metagenome""",281,20619243718


In [105]:
# # Let's go batch14_possible, split into 3Gbp chunks
# # Use the bases column because it is much more accurate than mbases
# batch15_possible = batch14_possible.join(df.select('acc','avespotlen','bases'), on="acc", how="inner")
# batch15_possible = batch15_possible.with_columns((pl.col('mbases') / 1e3).alias('gbases'))
# gbp_per_chunk = 3
# batch15_possible = batch15_possible.with_columns((gbp_per_chunk / pl.col('avespotlen') * 1e9).ceil().cast(pl.Int32).alias('chunk_size'))
# # batch16_possible = batch16_possible.with_columns((pl.col('gbases') / gbp_per_chunk).alias('num_chunks'))
# batch15_possible = batch15_possible.with_columns((pl.col('bases') / (pl.col('chunk_size') * pl.col('avespotlen'))).ceil().cast(pl.Int32).alias('num_chunks').cast(pl.Int32))
# batch15_possible.shape, batch15_possible[:4]

In [107]:
# # Ensure that avespotlen * chunk_size * num_chunks is > bases
# batch15_possible.filter(pl.col('chunk_size') * pl.col('avespotlen') * pl.col('num_chunks') < pl.col('bases'))

In [ ]:
# batch15_possible_exploded = batch15_possible.group_by('acc','avespotlen','num_chunks','gbases','chunk_size').agg(pl.arange(1, pl.col('num_chunks')+1).alias('chunk')).explode('chunk')


In [ ]:
# batch15_possible_exploded.sample(fraction=1, shuffle=True).write_csv('runlists/batch15_3gbp_chunks.txt')

# batch 16 - 15gbp chunks

In [ ]:
# # Let's go batch14_possible, split into 3Gbp chunks
# # Use the bases column because it is much more accurate than mbases
# batch16_possible = batch14_possible.join(df.select('acc','avespotlen','bases'), on="acc", how="inner")
# batch16_possible = batch16_possible.with_columns((pl.col('mbases') / 1e3).alias('gbases'))
# gbp_per_chunk = 15
# batch16_possible = batch16_possible.with_columns((gbp_per_chunk / pl.col('avespotlen') * 1e9).ceil().cast(pl.Int32).alias('chunk_size'))
# batch16_possible = batch16_possible.with_columns((pl.col('bases') / (pl.col('chunk_size') * pl.col('avespotlen'))).ceil().cast(pl.Int32).alias('num_chunks').cast(pl.Int32))
# batch16_possible.shape, batch16_possible[:4]

((3075, 9),
 shape: (4, 9)
 ┌─────────────┬─────────────┬────────┬────────┬───┬────────────┬─────────┬────────────┬────────────┐
 │ acc         ┆ releasedate ┆ mbases ┆ mbytes ┆ … ┆ bases      ┆ gbases  ┆ chunk_size ┆ num_chunks │
 │ ---         ┆ ---         ┆ ---    ┆ ---    ┆   ┆ ---        ┆ ---     ┆ ---        ┆ ---        │
 │ str         ┆ str         ┆ i64    ┆ i64    ┆   ┆ i64        ┆ f64     ┆ i32        ┆ i32        │
 ╞═════════════╪═════════════╪════════╪════════╪═══╪════════════╪═════════╪════════════╪════════════╡
 │ SRR5451739  ┆ 2017-04-17T ┆ 57300  ┆ 25291  ┆ … ┆ 5730089170 ┆ 57.3    ┆ 49668875   ┆ 4          │
 │             ┆ 00:00:00+00 ┆        ┆        ┆   ┆ 4          ┆         ┆            ┆            │
 │             ┆ :00         ┆        ┆        ┆   ┆            ┆         ┆            ┆            │
 │ SRR9008757  ┆ 2019-05-04T ┆ 68571  ┆ 20772  ┆ … ┆ 6857192913 ┆ 68.571  ┆ 49668875   ┆ 5          │
 │             ┆ 00:00:00+00 ┆        ┆        ┆   ┆ 6 

In [ ]:
# batch16_possible_exploded = batch16_possible.group_by('acc','avespotlen','num_chunks','gbases','chunk_size').agg(pl.arange(1, pl.col('num_chunks')+1).alias('chunk')).explode('chunk')
# batch16_possible_exploded[:10], batch16_possible_exploded.shape

(shape: (10, 6)
 ┌─────────────┬────────────┬────────────┬────────┬────────────┬───────┐
 │ acc         ┆ avespotlen ┆ num_chunks ┆ gbases ┆ chunk_size ┆ chunk │
 │ ---         ┆ ---        ┆ ---        ┆ ---    ┆ ---        ┆ ---   │
 │ str         ┆ i64        ┆ i32        ┆ f64    ┆ i32        ┆ i64   │
 ╞═════════════╪════════════╪════════════╪════════╪════════════╪═══════╡
 │ SRR30025521 ┆ 298        ┆ 3          ┆ 30.36  ┆ 50335571   ┆ 1     │
 │ SRR30025521 ┆ 298        ┆ 3          ┆ 30.36  ┆ 50335571   ┆ 2     │
 │ SRR30025521 ┆ 298        ┆ 3          ┆ 30.36  ┆ 50335571   ┆ 3     │
 │ ERR11547417 ┆ 302        ┆ 3          ┆ 32.291 ┆ 49668875   ┆ 1     │
 │ ERR11547417 ┆ 302        ┆ 3          ┆ 32.291 ┆ 49668875   ┆ 2     │
 │ ERR11547417 ┆ 302        ┆ 3          ┆ 32.291 ┆ 49668875   ┆ 3     │
 │ SRR26124403 ┆ 302        ┆ 7          ┆ 94.007 ┆ 49668875   ┆ 1     │
 │ SRR26124403 ┆ 302        ┆ 7          ┆ 94.007 ┆ 49668875   ┆ 2     │
 │ SRR26124403 ┆ 302        ┆ 7    

In [ ]:
# batch16_possible_exploded.sample(fraction=1, shuffle=True).write_csv('runlists/batch16_15gbp_chunks.txt')

# batch 17 - missing runs < 30Gbp

In [109]:
df = df.with_columns(pl.col('releasedate').str.slice(0,10).str.to_date("%Y-%m-%d"))
# 15/11/2024 is when Josh started his stuff, so we need everything before that
df2 = df.filter(pl.col('releasedate') <= pl.date(2024,11,15))

SchemaError: invalid series dtype: expected `String`, got `date`

In [110]:
# # ben in 🌐 b2 in singlem-sra-processing/cloud/argo on  main [!?] via 🐍 v3.12.3 via singlem-dev 20250325 took 8s
# # ➜  seq 9 |parallel rclone ls s3:woodcrob-sandpiper-us-east-1/unannotated{}/ '>' finished_lists/unannotated{}.ls
# #
# # ben in 🌐 b2 in singlem-sra-processing/cloud/argo on  main [?] via 🐍 v3.11.3 via mybase 20250325 took 4s
# # ➜  rclone ls s3://sandpiper-woodcrob/test/ >finished_lists/test.ls
# extern.run('cat finished_lists/*.ls  |grep RR |sed "s/  */\t/g" >/tmp/runs')

# prev10 = pl.read_csv('/tmp/runs', has_header=False, separator='\t')
# prev10.columns = ["null", "size", "path"]
# prev10 = prev10.with_columns(pl.col("path").alias("acc").str.split('.').list.get(0))

# finished = prev10.filter(pl.col('size') > 0)
# print('num zero size: {}'.format(prev10.filter(pl.col('size') == 0).shape[0]))
# print('num non-zero size: {}'.format(finished.shape[0]))

# # anti merge to total list
# unfinished = df2.filter(pl.col('mbases') < 30000).join(finished, on='acc', how='anti')
# unfinished.shape, unfinished.sample(4), unfinished.select(pl.col('mbytes').sum()) / 1e6 #=> 252 TB, too much to do locally

In [111]:
# num zero size: 7261
# num non-zero size: 633491
# ((144571, 7),
#  shape: (4, 7)
#  ┌─────────────┬─────────────┬────────┬────────┬──────────────────────────┬────────────┬────────────┐
#  │ acc         ┆ releasedate ┆ mbases ┆ mbytes ┆ organism                 ┆ avespotlen ┆ bases      │
#  │ ---         ┆ ---         ┆ ---    ┆ ---    ┆ ---                      ┆ ---        ┆ ---        │
#  │ str         ┆ date        ┆ i64    ┆ i64    ┆ str                      ┆ i64        ┆ i64        │
#  ╞═════════════╪═════════════╪════════╪════════╪══════════════════════════╪════════════╪════════════╡
#  │ ERR4066059  ┆ 2020-05-02  ┆ 2396   ┆ 927    ┆ metagenome               ┆ 294        ┆ 2396764382 │
#  │ SRR16881809 ┆ 2022-07-01  ┆ 3887   ┆ 1170   ┆ Banana mild mosaic virus ┆ 278        ┆ 3887539084 │
#  │ SRR23911021 ┆ 2023-03-20  ┆ 4613   ┆ 2991   ┆ feces metagenome         ┆ 202        ┆ 4613420228 │
#  │ SRR1519050  ┆ 2014-10-22  ┆ 3749   ┆ 2438   ┆ human metagenome         ┆ 101        ┆ 3749779227 │
#  └─────────────┴─────────────┴────────┴────────┴──────────────────────────┴────────────┴────────────┘,
#  shape: (1, 1)
#  ┌────────────┐
#  │ mbytes     │
#  │ ---        │
#  │ f64        │
#  ╞════════════╡
#  │ 252.189613 │
#  └────────────┘)

In [112]:
# unfinished.sort('releasedate')

In [ ]:
# Just run them I think, and keep the logs so we can see what failed.
# unfinished.sample(fraction=1, shuffle=True).select('acc').write_csv('runlists/batch17_30gbp_minus_finished.txt', include_header=False)

# Taking stock. Nearly there?

In [133]:
# Read in all finished lists 
extern.run('cat finished_lists/*.ls  |grep RR |sed "s/  */\t/g" >/tmp/runs')

prev11 = pl.read_csv('/tmp/runs', has_header=False, separator='\t')
prev11.columns = ["null", "size", "path"]
prev11 = prev11.with_columns(pl.col("path").alias("acc").str.split('.').list.get(0))

finished = prev11.filter(pl.col('size') > 0)
print('num zero size: {}'.format(prev11.filter(pl.col('size') == 0).shape[0]))
print('num non-zero size: {}'.format(finished.shape[0]))

# How many unchunked runs are finished?
print(
    finished.filter(pl.col('path').str.contains('chunk') == False).group_by('acc').len().shape[0], df.shape[0]
)
show_all(finished.filter(pl.col('path').str.contains('chunk')).sample(5))

# anti merge to total list
# unfinished = df2.filter(pl.col('mbases') < 30000).join(finished, on='acc', how='anti')
# unfinished.shape, unfinished.sample(4), unfinished.select(pl.col('mbytes').sum()) / 1e6 #=> 252 TB, too much to do locally

num zero size: 11207
num non-zero size: 905060
777076 783205
shape: (5, 4)
┌──────┬─────────┬─────────────────────────────────────────────────────────────┬─────────────┐
│ null ┆ size    ┆ path                                                        ┆ acc         │
│ ---  ┆ ---     ┆ ---                                                         ┆ ---         │
│ str  ┆ i64     ┆ str                                                         ┆ str         │
╞══════╪═════════╪═════════════════════════════════════════════════════════════╪═════════════╡
│ null ┆ 45709   ┆ SRR31168975.size49668875.chunk3.unannotated.singlem.json.gz ┆ SRR31168975 │
│ null ┆ 2221560 ┆ SRR23978799.size49668875.chunk1.unannotated.singlem.json.gz ┆ SRR23978799 │
│ null ┆ 2079342 ┆ SRR18078965.size53571429.chunk5.unannotated.singlem.json.gz ┆ SRR18078965 │
│ null ┆ 2191536 ┆ SRR27584587.size49668875.chunk6.unannotated.singlem.json.gz ┆ SRR27584587 │
│ null ┆ 2237886 ┆ SRR20336965.size56818182.chunk4.unannotated.singlem

In [130]:
# Which ones are done by chunk?

# Original code:

# # Let's go batch14_possible, split into 3Gbp chunks
# # Use the bases column because it is much more accurate than mbases
# batch16_possible = batch14_possible.join(df.select('acc','avespotlen','bases'), on="acc", how="inner")
# batch16_possible = batch16_possible.with_columns((pl.col('mbases') / 1e3).alias('gbases'))
# gbp_per_chunk = 15
# batch16_possible = batch16_possible.with_columns((gbp_per_chunk / pl.col('avespotlen') * 1e9).ceil().cast(pl.Int32).alias('chunk_size'))
# batch16_possible = batch16_possible.with_columns((pl.col('bases') / (pl.col('chunk_size') * pl.col('avespotlen'))).ceil().cast(pl.Int32).alias('num_chunks').cast(pl.Int32))
# batch16_possible.shape, batch16_possible[:4]

chunked_accs = finished.filter(pl.col('path').str.contains('chunk')).group_by('acc').len()
chunked_accs = chunked_accs.join(df2, on='acc', how='inner')

gbp_per_chunk = 15
batch16_possible = chunked_accs.with_columns((gbp_per_chunk / pl.col('avespotlen') * 1e9).ceil().cast(pl.Int32).alias('chunk_size'))
batch16_possible = batch16_possible.with_columns((pl.col('bases') / (pl.col('chunk_size') * pl.col('avespotlen'))).ceil().cast(pl.Int32).alias('num_chunks').cast(pl.Int32))
batch16_possible.shape, batch16_possible[:4]

batch16_possible.filter(pl.col('len')==pl.col('num_chunks')).shape[0], batch16_possible.shape[0]

(2141, 2895)

In [137]:
regular_finishes = finished.filter(pl.col('path').str.contains('chunk') == False)['acc']
chunked_finishes = batch16_possible.filter(pl.col('len')==pl.col('num_chunks'))['acc']
# create a list of all the finished runs
all_finished = pl.concat([regular_finishes, chunked_finishes])
all_finished = all_finished.unique()
all_finished.shape[0], all_finished[:5], df.shape[0], all_finished.shape[0]/df.shape[0]*100

(779214,
 shape: (5,)
 Series: 'acc' [str]
 [
 	"SRR19577162"
 	"ERR2001159"
 	"SRR27217407"
 	"SRR30025163"
 	"ERR13958464"
 ],
 783205,
 99.49042715508712)

In [ ]:
# => So 99.49% of the runs are finished, and 0.51% are not. I think I might leave the rest until later, after holidays. Likely the idea would be just to download the remaining runs, to get better control and not have to monitor the cloud so much.

783205